# Pipeline de Detección de Fraude — Demo

Este notebook ejecuta el pipeline completo (Bronze → Silver → Gold) sobre datos sintéticos de transacciones bancarias.

**Arquitectura**: AWS Glue (PySpark) + Delta Lake + Airflow

**Stack**: Medallion (Bronze/Silver/Gold)

## 1. Configuración e inicio de Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import delta
import os, json, random, uuid
from datetime import datetime, timedelta

builder = SparkSession.builder \
    .appName("Fraude_Demo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = delta.configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark iniciado:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/30 16:55:12 WARN Utils: Your hostname, MacBook-Air-de-Adrian.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.85 instead (on interface en0)
26/06/30 16:55:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/adrianespinoza/anaconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/adrianespinoza/.ivy2.5.2/cache
The jars for the packages stored in: /Users/adrianespinoza/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7bceab9b-db68-4da1-8070-ba2b543d2724;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.0 in central
	found io.delta#delta-storage;4.3.0 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf

	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.3.0 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.3.0 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.

26/06/30 16:55:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark iniciado: 4.1.1


## 2. Generación de datos sintéticos

Generamos 50.000 transacciones sintéticas que replican la estructura de datos bancarios:
- `transaction_id`: identificador único
- `user_id`: UUID del cliente
- `card`: número de tarjeta (16 dígitos)
- `amount`: monto (con algunos valores ≤0 para probar el filtro de calidad)
- `transaction_date`: fecha de la transacción
- `timestamp`: datetime completo
- `merchant`: comercio
- `currency`: moneda

In [2]:
ROWS = 50_000
USERS = [str(uuid.uuid4()) for _ in range(500)]
MERCHANTS = ["Amazon","Netflix","Spotify","Uber","Rappi","Falabella","MercadoLibre","Disney+","Apple","Google"]
CARDS = ["4532015112890367","4916123456789012","4556123456789013","4929123456789014","4539123456789015",
         "4485123456789016","4716123456789017","4024123456789018","4532123456789019","4913123456789020"]
CURRENCIES = ["USD","EUR","CLP","BRL"]

start = datetime(2026, 1, 1)
rows = []
for i in range(ROWS):
    amount = round(abs(random.gauss(150, 300)) + 0.01, 2)
    if i % 75 == 0:
        amount = random.choice([-amount, 0.0])
    tx_date = start + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23))
    rows.append({
        "transaction_id": f"demo-{i:06d}",
        "user_id": random.choice(USERS),
        "card": random.choice(CARDS),
        "amount": amount,
        "transaction_date": tx_date.strftime("%Y-%m-%d"),
        "timestamp": tx_date.strftime("%Y-%m-%dT%H:%M:%S"),
        "merchant": random.choice(MERCHANTS),
        "currency": random.choice(CURRENCIES),
    })

df_raw = spark.createDataFrame(rows)
print(f"Filas generadas: {df_raw.count():,}")
df_raw.show(5, truncate=False)

Filas generadas: 50,000
+-------+----------------+--------+---------+-------------------+----------------+--------------+------------------------------------+
|amount |card            |currency|merchant |timestamp          |transaction_date|transaction_id|user_id                             |
+-------+----------------+--------+---------+-------------------+----------------+--------------+------------------------------------+
|-356.99|4913123456789020|USD     |Disney+  |2026-03-07T21:00:00|2026-03-07      |demo-000000   |6c92c565-3781-4f7e-b227-3b57d3734a18|
|291.76 |4716123456789017|USD     |Uber     |2026-06-23T13:00:00|2026-06-23      |demo-000001   |45d84a32-abf2-4d41-87a0-e60ac4ffd23f|
|258.54 |4929123456789014|CLP     |Google   |2026-02-11T13:00:00|2026-02-11      |demo-000002   |cca952f8-5128-4a2b-99df-2107cabecb0c|
|284.0  |4913123456789020|EUR     |Falabella|2026-06-19T15:00:00|2026-06-19      |demo-000003   |c3a4c58d-5b0e-4646-9ffa-b0b0d0c8fb4c|
|312.11 |4024123456789018|CLP  

Traceback (most recent call last):
  File "/Users/adrianespinoza/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/Users/adrianespinoza/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


## 3. Capa Bronze — Ingesta inmutable

Agregamos metadatos de linaje (`_ingestion_ts`, `_source_file`) y escribimos en formato Parquet.

**Decisión arquitectónica**: Bronze en Parquet simple (no Delta) porque es una capa de staging inmutable. Si se requiere replay, se relee desde la fuente original.

In [3]:
BRONZE_PATH = "/tmp/demo/bronze/transactions/"

df_bronze = df_raw.withColumn("_ingestion_ts", F.current_timestamp()) \
                  .withColumn("_source_file", F.lit("demo_generated"))

df_bronze.write.mode("overwrite").parquet(BRONZE_PATH)

df_bronze_check = spark.read.parquet(BRONZE_PATH)
print(f"Bronze: {df_bronze_check.count():,} filas escritas en Parquet")
df_bronze_check.select("transaction_id", "amount", "_ingestion_ts", "_source_file").show(5, truncate=False)

26/06/30 16:55:17 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/06/30 16:55:17 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/06/30 16:55:17 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 76,00% for 10 writers


26/06/30 16:55:18 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/06/30 16:55:18 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Bronze: 50,000 filas escritas en Parquet
+--------------+------+--------------------------+--------------+
|transaction_id|amount|_ingestion_ts             |_source_file  |
+--------------+------+--------------------------+--------------+
|demo-004096   |186.71|2026-06-30 16:55:17.269367|demo_generated|
|demo-004097   |306.08|2026-06-30 16:55:17.269367|demo_generated|
|demo-004098   |54.13 |2026-06-30 16:55:17.269367|demo_generated|
|demo-004099   |387.61|2026-06-30 16:55:17.269367|demo_generated|
|demo-004100   |259.14|2026-06-30 16:55:17.269367|demo_generated|
+--------------+------+--------------------------+--------------+
only showing top 5 rows


## 4. Capa Silver — Calidad y enmascaramiento PII

Transformaciones aplicadas:
1. **Filtro de calidad**: solo transacciones con `amount > 0`
2. **Enmascaramiento PII**: el número de tarjeta se reemplaza por `4532-XXXX-XXXX-0367`, protegiendo datos sensibles
3. Escritura en **Delta Lake** con particionamiento por `transaction_date`

**Decisión arquitectónica**: Silver en Delta Lake para aprovechar ACID transactions, time travel, y compactación.

In [4]:
SILVER_PATH = "/tmp/demo/silver/transactions/"

df_silver = df_bronze.filter(F.col("amount") > 0) \
    .withColumn("card_masked",
                F.concat(F.substring(F.col("card"), 1, 4),
                         F.lit("-XXXX-XXXX-"),
                         F.substring(F.col("card"), -4, 4))) \
    .drop("card")

df_silver.write.format("delta").mode("overwrite") \
    .partitionBy("transaction_date").save(SILVER_PATH)

df_silver_check = spark.read.format("delta").load(SILVER_PATH)
total = df_silver_check.count()
invalid = ROWS - total
print(f"Silver: {total:,} filas válidas (se eliminaron {invalid} filas con amount ≤ 0)")
df_silver_check.select("transaction_id", "amount", "card_masked").show(5, truncate=False)

26/06/30 16:55:19 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


26/06/30 16:55:20 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


26/06/30 16:55:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Silver: 49,333 filas válidas (se eliminaron 667 filas con amount ≤ 0)


+--------------+------+-------------------+
|transaction_id|amount|card_masked        |
+--------------+------+-------------------+
|demo-014894   |50.25 |4716-XXXX-XXXX-9017|
|demo-014898   |393.59|4485-XXXX-XXXX-9016|
|demo-015013   |298.11|4532-XXXX-XXXX-0367|
|demo-015314   |150.63|4556-XXXX-XXXX-9013|
|demo-015658   |292.48|4913-XXXX-XXXX-9020|
+--------------+------+-------------------+
only showing top 5 rows


### Verificación de enmascaramiento

Confirmamos que no existen números de tarjeta completos en la capa Silver.

In [5]:
card_cols = [c for c in df_silver_check.columns if "card" in c.lower()]
print(f"Columnas relacionadas a tarjeta: {card_cols}")
assert "card" not in df_silver_check.columns, "ERROR: columna 'card' aún existe en Silver"
print("OK: columna 'card' eliminada correctamente.")

Columnas relacionadas a tarjeta: ['card_masked']
OK: columna 'card' eliminada correctamente.


## 5. Capa Gold — Agregación de riesgo

Agregamos transacciones por `user_id` y `transaction_date` para generar un perfil de riesgo diario.

**Heurística**: un usuario se marca como `is_high_risk = True` cuando realiza más de 12 transacciones en un mismo día.

**Decisión arquitectónica**: Gold en Delta Lake para servir como fuente confiable a dashboards y modelos ML.

In [6]:
GOLD_PATH = "/tmp/demo/gold/risk_profile/"

df_gold = df_silver_check.groupBy("user_id", "transaction_date") \
    .agg(F.count("*").alias("tx_count"),
         F.sum("amount").alias("daily_total")) \
    .withColumn("is_high_risk", F.col("tx_count") > 12)

df_gold.write.format("delta").mode("overwrite").save(GOLD_PATH)

df_gold_check = spark.read.format("delta").load(GOLD_PATH)
print(f"Gold: {df_gold_check.count():,} registros de perfil de riesgo")
df_gold_check.orderBy(F.col("tx_count").desc()).show(10, truncate=False)

Gold: 37,986 registros de perfil de riesgo


+------------------------------------+----------------+--------+------------------+------------+
|user_id                             |transaction_date|tx_count|daily_total       |is_high_risk|
+------------------------------------+----------------+--------+------------------+------------+
|c92fe5ba-e8fe-4465-9182-c8d205ab02e2|2026-02-03      |6       |1631.22           |false       |
|72ecb0fe-7cd8-48f8-9f18-1f7ffeffd924|2026-02-22      |6       |1871.58           |false       |
|8e5de67a-5b0f-4f26-94f0-eaad53d46879|2026-04-06      |6       |1348.44           |false       |
|4e800226-110b-440a-9339-b814eaa61a0f|2026-05-02      |5       |844.54            |false       |
|67e18730-984b-4e0e-b3f4-8f778dd2cfca|2026-03-11      |5       |1649.65           |false       |
|a4a2e083-3f5d-4ce8-9f7b-f6e443a8a7c4|2026-01-12      |5       |1447.4            |false       |
|3abe58e5-1213-4f32-b68e-4a9d94ac4780|2026-05-27      |5       |1081.3799999999999|false       |
|bac1f98d-d2a9-4d5b-b58c-44cb3

## 6. Resultados

Resumen del pipeline completo.

In [7]:
high_risk = df_gold_check.filter("is_high_risk = true").count()
total_users = df_gold_check.select("user_id").distinct().count()
print("=" * 50)
print("RESULTADOS DEL PIPELINE")
print("=" * 50)
print(f"  Bronze: {ROWS:,} filas raw")
print(f"  Silver: {total:,} filas válidas ({(total/ROWS)*100:.1f}% retención)")
print(f"  Gold:   {df_gold_check.count():,} perfiles de riesgo")
print(f"  Usuarios únicos: {total_users:,}")
print(f"  Usuarios high-risk (>12 tx/día): {high_risk:,}")
print("=" * 50)

print("\nTop 5 usuarios con más transacciones en un día:")
df_gold_check.orderBy(F.col("tx_count").desc()).select(
    "user_id", "transaction_date", "tx_count", "daily_total", "is_high_risk"
).show(5, truncate=False)

RESULTADOS DEL PIPELINE
  Bronze: 50,000 filas raw
  Silver: 49,333 filas válidas (98.7% retención)
  Gold:   37,986 perfiles de riesgo
  Usuarios únicos: 500
  Usuarios high-risk (>12 tx/día): 0

Top 5 usuarios con más transacciones en un día:
+------------------------------------+----------------+--------+-----------+------------+
|user_id                             |transaction_date|tx_count|daily_total|is_high_risk|
+------------------------------------+----------------+--------+-----------+------------+
|8e5de67a-5b0f-4f26-94f0-eaad53d46879|2026-04-06      |6       |1348.44    |false       |
|72ecb0fe-7cd8-48f8-9f18-1f7ffeffd924|2026-02-22      |6       |1871.58    |false       |
|c92fe5ba-e8fe-4465-9182-c8d205ab02e2|2026-02-03      |6       |1631.22    |false       |
|fc555e55-2f85-4c7b-8d95-1428dbaa827f|2026-04-25      |5       |958.24     |false       |
|b4653f72-a37c-4f87-b644-e6c8488a8b6a|2026-04-08      |5       |2313.24    |false       |
+----------------------------------

## 7. Limpieza

Eliminamos datos temporales del demo.

In [8]:
import shutil
for p in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    shutil.rmtree(p, ignore_errors=True)
spark.stop()
print("Demo completado. Spark detenido.")

Demo completado. Spark detenido.
